# PU Learning — Kaggle Notebook

PyTorch による **Positive-Unlabeled (PU) 学習** の実行スクリプトです。  
論文: [https://arxiv.org/abs/2107.05045](https://arxiv.org/abs/2107.05045)

---

## 事前準備: プロジェクトファイルのアップロード

1. 右サイドバーの **"Add data"** → **"Upload"** を選択
2. 以下のファイルを **全て** アップロード (フォルダ構造を維持):
   ```
   main.py
   algorithm.py
   dataset.py
   model.py
   metric.py
   save.py
   run_benchmark.py
   run_synthetic.py
   modules/Kernel_MPE.py
   ```
3. アップロード時のデータセット名 (slug) を控えておく (例: `pu-learning-src`)
4. **Accelerator** を **GPU T4** に設定

> **注意**: `/kaggle/input/` は読み取り専用です。ファイルは `/kaggle/working/` にコピーして使用します。

In [ ]:
# ===== 環境確認 & 依存パッケージのインストール =====
import subprocess, sys, os

# GPU確認
print("=== GPU Info ===")
!nvidia-smi

print("\n=== PyTorch & CUDA ===")
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

# cvxopt のインストール (Kernel_MPE.py が依存)
print("\n=== Installing cvxopt ===")
!pip install cvxopt -q
print("cvxopt installed.")

In [ ]:
# ===== プロジェクトファイルのコピー =====
import shutil, glob

WORKING_DIR = "/kaggle/working"
INPUT_DIR   = "/kaggle/input"

# アップロードしたデータセットのスラッグを自動検出
input_datasets = os.listdir(INPUT_DIR)
print(f"Available datasets in /kaggle/input: {input_datasets}")

# プロジェクトファイルが含まれるデータセットを特定
project_src = None
for ds in input_datasets:
    candidate = os.path.join(INPUT_DIR, ds)
    if os.path.exists(os.path.join(candidate, "main.py")):
        project_src = candidate
        break

if project_src is None:
    raise FileNotFoundError(
        "main.py が見つかりません。\n"
        "右サイドバーの 'Add data' からプロジェクトファイルをアップロードしてください。"
    )

print(f"Project source  : {project_src}")

# 必要なファイルを /kaggle/working にコピー
FILES_TO_COPY = [
    "main.py", "algorithm.py", "dataset.py", "model.py",
    "metric.py", "save.py", "run_benchmark.py", "run_synthetic.py",
]
for fname in FILES_TO_COPY:
    src = os.path.join(project_src, fname)
    dst = os.path.join(WORKING_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  Copied: {fname}")
    else:
        print(f"  WARNING: {fname} not found in {project_src}")

# modules/ ディレクトリのコピー
modules_src = os.path.join(project_src, "modules")
modules_dst = os.path.join(WORKING_DIR, "modules")
os.makedirs(modules_dst, exist_ok=True)
for fpath in glob.glob(os.path.join(modules_src, "*.py")):
    dst = os.path.join(modules_dst, os.path.basename(fpath))
    shutil.copy2(fpath, dst)
    print(f"  Copied: modules/{os.path.basename(fpath)}")

# modules/__init__.py がなければ作成
init_path = os.path.join(modules_dst, "__init__.py")
if not os.path.exists(init_path):
    open(init_path, "w").close()

print("\nFile setup complete.")

In [ ]:
# ===== Kaggle 向けパッチ: DataLoader の num_workers を 0 に設定 =====
# Kaggle のマルチプロセス制限に対応するため num_workers を 0 に変更
import re

def patch_num_workers(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    patched = re.sub(r"num_workers=\d+", "num_workers=0", content)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(patched)
    print(f"  Patched: {os.path.basename(filepath)}")

print("Applying num_workers=0 patch...")
patch_num_workers(os.path.join(WORKING_DIR, "run_benchmark.py"))
patch_num_workers(os.path.join(WORKING_DIR, "run_synthetic.py"))
print("Patch applied.")

## 実験設定

下のセルで実験パラメータを設定してください。  
変更が必要なのは **この設定セルだけ** です。

| パラメータ | 選択肢 | 説明 |
|-----------|--------|------|
| `METHOD`  | `uPU` / `nnPU` / `PUa` / `DRPU` | 学習アルゴリズム |
| `DATASET` | `gauss` / `gauss_mix` / `mnist` / `fmnist` / `kmnist` / `cifar` | データセット |
| `USE_PRESET` | `True` / `False` | プリセットパラメータを使用 |
| `GPU_ID`  | `0` (GPU) / `-1` (CPU) | 使用デバイス |
| `SEED`    | 整数 or `None` | 乱数シード |

In [ ]:
# ===== 実験設定 (ここを変更してください) =====

METHOD      = "DRPU"    # uPU / nnPU / PUa / DRPU
DATASET     = "mnist"   # gauss / gauss_mix / mnist / fmnist / kmnist / cifar
USE_PRESET  = True      # True: プリセット設定を使用 / False: 以下の詳細設定を使用
GPU_ID      = 0         # 0: GPU T4 / -1: CPU
SEED        = 42        # 再現性のための乱数シード

# USE_PRESET = False の場合のみ有効
NUM_POSITIVE = 2500     # 正例ラベルデータ数
MAX_EPOCHS   = 50       # 学習エポック数
BATCH_SIZE   = 500      # バッチサイズ
LR           = 2e-5     # 学習率

# 出力ディレクトリ
RES_DIR  = "/kaggle/working/results"
DATA_DIR = "/kaggle/working/dataset"
JOB_ID   = 0

# ==================================================
import sys
sys.path.insert(0, WORKING_DIR)

print(f"Method  : {METHOD}")
print(f"Dataset : {DATASET}")
print(f"Preset  : {USE_PRESET}")
print(f"GPU ID  : {GPU_ID}")
print(f"Seed    : {SEED}")

In [ ]:
# ===== 学習の実行 =====
import subprocess, time

cmd = [
    sys.executable, os.path.join(WORKING_DIR, "main.py"),
    "--method",  METHOD,
    "--dataset", DATASET,
    "--gpu",     str(GPU_ID),
    "--res_dir", RES_DIR,
    "--data_dir", DATA_DIR,
    "--seed",    str(SEED),
    "--id",      str(JOB_ID),
]

if USE_PRESET:
    cmd.append("--preset")
else:
    cmd += [
        "--num_positive", str(NUM_POSITIVE),
        "--max_epochs",   str(MAX_EPOCHS),
        "--batch_size",   str(BATCH_SIZE),
        "--lr",           str(LR),
    ]

print("Running command:")
print(" ".join(cmd))
print("-" * 60)

start = time.time()
result = subprocess.run(
    cmd,
    cwd=WORKING_DIR,
    capture_output=False,  # stdout/stderr をリアルタイム表示
)
elapsed = time.time() - start

print("-" * 60)
if result.returncode == 0:
    print(f"Training complete. ({elapsed:.1f} sec)")
else:
    print(f"ERROR: return code = {result.returncode}")

In [ ]:
# ===== 結果の表示 =====
import pandas as pd

log_path = os.path.join(RES_DIR, METHOD, DATASET, f"log_{JOB_ID}.txt")

if os.path.exists(log_path):
    print(f"=== Log: {log_path} ===")
    with open(log_path, encoding="utf-8") as f:
        print(f.read())
else:
    print(f"Log not found: {log_path}")

# test-i/ ディレクトリから accuracy / AUC をまとめて表示
print("\n=== Test Results (per prior) ===")
rows = []
test_dir_base = os.path.join(RES_DIR, METHOD, DATASET)
for i in range(10):  # 最大 10 prior まで探索
    test_dir = os.path.join(test_dir_base, f"test-{i}")
    acc_file = os.path.join(test_dir, "accuracy.txt")
    auc_file = os.path.join(test_dir, "auc.txt")
    if not os.path.exists(acc_file):
        break
    with open(acc_file) as f:
        acc_vals = [float(v.strip()) for v in f.readlines() if v.strip()]
    with open(auc_file) as f:
        auc_vals = [float(v.strip()) for v in f.readlines() if v.strip()]
    rows.append({
        "Test Prior Index": i,
        "Accuracy": f"{acc_vals[-1]:.4f}" if acc_vals else "N/A",
        "AUC":      f"{auc_vals[-1]:.4f}" if auc_vals else "N/A",
    })

if rows:
    df = pd.DataFrame(rows)
    display(df)
else:
    print("No test results found.")

In [ ]:
# ===== ロスカーブの可視化 =====
import matplotlib.pyplot as plt
import numpy as np

history_dir = os.path.join(RES_DIR, METHOD, DATASET, "train", f"history_{JOB_ID}")

train_loss_path = os.path.join(history_dir, "train_loss.csv")
val_loss_path   = os.path.join(history_dir, "validation_loss.csv")

if os.path.exists(train_loss_path):
    train_loss = np.loadtxt(train_loss_path)
    val_loss   = np.loadtxt(val_loss_path) if os.path.exists(val_loss_path) else None

    epochs = np.arange(1, len(train_loss) + 1)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(epochs, train_loss, label="Train Loss", linewidth=1.5)
    if val_loss is not None:
        ax.plot(epochs, val_loss, label="Validation Loss", linewidth=1.5, linestyle="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"{METHOD} / {DATASET} — Loss Curve")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print(f"Loss CSV not found: {train_loss_path}")
    print("先に学習セルを実行してください。")

---
## (オプション) 複数メソッドの一括実行

全メソッド (`uPU`, `nnPU`, `PUa`, `DRPU`) を指定データセットで順番に実行します。  
Kaggle T4 GPU で MNIST なら目安 **20〜40 分** かかります。

In [ ]:
# ===== 複数メソッドの一括実行 =====
# 実行したいメソッドとデータセットをリストで指定
BATCH_METHODS  = ["uPU", "nnPU", "DRPU"]  # 実行するメソッド ("PUa" は処理時間長め)
BATCH_DATASET  = "mnist"                   # 全メソッドで使うデータセット
BATCH_SEED     = 42

results_summary = []

for idx, method in enumerate(BATCH_METHODS):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(BATCH_METHODS)}] Method: {method}, Dataset: {BATCH_DATASET}")
    print(f"{'='*60}")

    cmd = [
        sys.executable, os.path.join(WORKING_DIR, "main.py"),
        "--method",   method,
        "--dataset",  BATCH_DATASET,
        "--preset",
        "--gpu",      str(GPU_ID),
        "--res_dir",  RES_DIR,
        "--data_dir", DATA_DIR,
        "--seed",     str(BATCH_SEED),
        "--id",       str(idx),
    ]

    start = time.time()
    proc = subprocess.run(cmd, cwd=WORKING_DIR)
    elapsed = time.time() - start

    status = "OK" if proc.returncode == 0 else f"ERROR({proc.returncode})"
    results_summary.append({"Method": method, "Status": status, "Time (s)": f"{elapsed:.0f}"})
    print(f"Done: {method} — {status} ({elapsed:.0f}s)")

print("\n=== Batch Run Summary ===")
display(pd.DataFrame(results_summary))

In [ ]:
# ===== 一括実行結果の比較表示 =====
print("=== Accuracy & AUC Comparison ===")

compare_rows = []
for idx, method in enumerate(BATCH_METHODS):
    test_dir_base = os.path.join(RES_DIR, method, BATCH_DATASET)
    method_accs, method_aucs = [], []
    for i in range(10):
        acc_file = os.path.join(test_dir_base, f"test-{i}", "accuracy.txt")
        auc_file = os.path.join(test_dir_base, f"test-{i}", "auc.txt")
        if not os.path.exists(acc_file):
            break
        with open(acc_file) as f:
            vals = [float(v.strip()) for v in f.readlines() if v.strip()]
            if vals:
                method_accs.append(vals[-1])
        with open(auc_file) as f:
            vals = [float(v.strip()) for v in f.readlines() if v.strip()]
            if vals:
                method_aucs.append(vals[-1])
    if method_accs:
        compare_rows.append({
            "Method":       method,
            "Avg Accuracy": f"{np.mean(method_accs):.4f}",
            "Avg AUC":      f"{np.mean(method_aucs):.4f}" if method_aucs else "N/A",
        })

if compare_rows:
    display(pd.DataFrame(compare_rows))

# ロスカーブの比較プロット
fig, ax = plt.subplots(figsize=(9, 4))
for idx, method in enumerate(BATCH_METHODS):
    loss_path = os.path.join(RES_DIR, method, BATCH_DATASET, "train", f"history_{idx}", "train_loss.csv")
    if os.path.exists(loss_path):
        loss = np.loadtxt(loss_path)
        ax.plot(np.arange(1, len(loss)+1), loss, label=method, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train Loss")
ax.set_title(f"Training Loss Comparison — {BATCH_DATASET}")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()